In [32]:
import numpy as np
import pandas as pd
import ast

In [33]:
train_df= pd.read_csv("train_data.csv")
test_df= pd.read_csv("test_data.csv")

In [34]:
train_df['pixels']= train_df['pixels'].apply(ast.literal_eval).apply(lambda x: np.array(x, dtype= np.float32))
test_df['pixels']= test_df['pixels'].apply(ast.literal_eval).apply(lambda x: np.array(x, dtype= np.float32))

mean_values= np.mean(train_df['pixels'])

train_df['pixels']= train_df['pixels'].apply(lambda x: np.subtract(x, mean_values))


test_df['pixels']= test_df['pixels'].apply(lambda x: np.subtract(x, mean_values))

X_train= np.stack(train_df['pixels'].values)
y= train_df['class']
X_test= np.stack(test_df['pixels'].values)

In [35]:
answer= []

for _, row in train_df.iterrows():
    answer.append({
        'subtaskID': 1,
        'datapointID': row['id'],
        "answer": row['pixels']
    })

In [ ]:
from sklearn.svm import LinearSVC
from sklearn.utils.class_weight import compute_class_weight
from sklearn.decomposition import PCA

pca= PCA(n_components=200)
X_train= pca.fit_transform(X_train)
X_test= pca.transform(X_test)

class_weights= dict(zip(np.unique(y),compute_class_weight(class_weight='balanced', classes= np.unique(y), y=y)))

model= LinearSVC(max_iter=10000, random_state=42, class_weight=class_weights, C= 0.1)

model.fit(X_train, y)

predictions= model.predict(X_test)

for id_, pred in zip(test_df['id'], predictions):
    answer.append({
        'subtaskID': 2,
        'datapointID': id_,
        "answer": pred
    })

In [37]:
pd.DataFrame(answer).to_csv("submission.csv", index= False)